# 01 数据预处理与上传 HDFS

读取 2016-2020 Excel，转换为 CSV 并上传 HDFS。

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path('..')
DATA_DIR = BASE / 'data'
CSV_DIR = DATA_DIR / 'csv'
CSV_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
files = sorted(DATA_DIR.glob('*.xlsx'))
frames = []
for f in files:
    year = f.stem[:4]
    df = pd.read_excel(f, sheet_name=0)
    if len(df) > 0 and str(df.iloc[0].get('EventClassify', '')).strip() == '事件分类':
        df = df.iloc[1:].copy()
    df['Year'] = int(year)
    df.to_csv(CSV_DIR / f'{year}.csv', index=False, encoding='utf-8-sig')
    frames.append(df)

all_df = pd.concat(frames, ignore_index=True)
all_df.to_csv(CSV_DIR / 'disaster_2016_2020.csv', index=False, encoding='utf-8-sig')
all_df.head()

In [ ]:
print('在 hadoop-namenode 容器执行：')
print('hdfs dfs -mkdir -p /data/disaster')
print('hdfs dfs -put -f /data/csv/disaster_2016_2020.csv /data/disaster/')